# Look at mapping breadth and depth of 100 x metagenomes to singleclust genes

In [1]:
import polars as pl
import glob
import os
import screed
import csv
import screed

## Read in individual metag x species singleclust depth reports

These files contain depth-of-mapping information for all our genes.

In [2]:
DIR='../outputs.cds/singleclust/bam.bak'
template = '../outputs.cds/singleclust/bam/{metag}.x.{species}.depth.txt'

def read_depth_txt(metag, species, *, exclude_ends=75):
    filename = template.format(metag=metag, species=species)
    df = pl.read_csv(filename, separator='\t', has_header=False,
                 new_columns=('gene', 'pos', 'cov', 'foo')).select(['gene', 'pos', 'cov'])

    sum_df = df.group_by('gene').all().with_columns(
        # select slice [75:-75]
        (pl.col("pos").list.slice(exclude_ends, -exclude_ends).list.len()).alias("len"),
        (pl.col("cov").list.slice(exclude_ends, -exclude_ends)),
        (pl.col("cov").list.slice(exclude_ends, -exclude_ends).list.filter(pl.element() > 0)).list.len().alias("hits"),
    ).with_columns(
        (pl.lit(metag).alias("metag")),
        (pl.lit(species).alias("species")),
        # summarize: average depth across contig,
        (pl.col("cov").list.sum() / pl.col("len")).alias("depth_all"),
        # average depth across covered bases,
        (pl.col("cov").list.sum() / pl.col("hits")).alias("depth_cov"),
        # fraction of bases covered
        (pl.col("hits") / pl.col("cov").list.len()).alias("breadth"),
    ).select(["metag", "species", "gene", "len", "hits", "breadth", "depth_all", "depth_cov"])
    return sum_df

read_depth_txt('ERR1135199', 's__Cryptobacteroides sp900546925')

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""MFHJPMDA_01033""",1137,1124,0.988566,6.783641,6.8621
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""EHLNOCPD_01770""",765,399,0.521569,1.028758,1.972431
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""FOKFIHIM_00492""",843,640,0.759193,1.381969,1.8203125
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""HEEBLBEE_01085""",312,181,0.580128,0.74359,1.281768
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""FNIGJHLI_02431""",2478,0,0.0,0.0,NaN
…,…,…,…,…,…,…,…
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""BEKEPKDC_00736""",3375,2554,0.756741,1.487704,1.965936
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""OIJDNDLK_01211""",2388,479,0.200586,0.239112,1.192067
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""FNIGJHLI_01523""",3009,0,0.0,0.0,NaN


In [3]:
# Read them all in!
filenames = glob.glob(f"{DIR}/*.depth.txt")

dflist = []
for i, n in enumerate(filenames):
    if i % 100 == 0:
        print(f"{i} of {len(filenames)}")
    n = os.path.basename(n)
    metag, _, species, _ = n.split('.', 3)
    dflist.append(read_depth_txt(metag, species))

depth_df = pl.concat(dflist)

print(f"read {len(filenames)} depth files.")

0 of 1400
100 of 1400
200 of 1400
300 of 1400
400 of 1400
500 of 1400
600 of 1400
700 of 1400
800 of 1400
900 of 1400
1000 of 1400
1100 of 1400
1200 of 1400
1300 of 1400
read 1400 depth files.


In [ ]:
depth_df.filter(pl.col('depth_all').is_not_nan()).sort(by='depth_all', descending=True).filter(pl.col('depth_all') > 0.0)

## Summarize our mapping breadth results across all the metagenomes

In [ ]:
# require 10% of each gene to be covered by at least one read
BREADTH_CUTOFF = 0.1

In [ ]:
# aggregate across all metagenomes;
# calculate fraction of metagenomes for which gene mapping exceeds our breadth cutoff
agg_df = depth_df.group_by(['species', 'gene']).agg(
    # get fraction of columns where breadth is greater than cutoff as 'f'
    ((pl.col("breadth") >= BREADTH_CUTOFF).sum() / pl.col("breadth").len()).alias("f"),
#    (pl.col("depth").filter(pl.col("depth").is_not_nan()).mean()),
)
agg_df

In [ ]:
# print information out by species
for species in sorted(agg_df['species'].unique()):
    print(species)
    foo_df = agg_df.filter(pl.col("species") == species)
    foo_df = foo_df.sort(by='f', descending=True).filter(pl.col('f') > 0.8)
    print(foo_df)

    top50_names = set(foo_df.head(50)['gene'].to_list())

    outfile = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.fa'
    print(outfile)
    outfp = open(outfile, 'wt')
    for record in screed.open(f'../outputs.cds/singleclust/{species}.cds3.min50.dedup.fa'):
        name = record.name.split(' ')[0]
        if name in top50_names:
            top50_names.remove(name)
            outfp.write(f'>{record.name}\n{record.sequence}\n')
    assert not top50_names
    outfp.close()

    outfile2 = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.csv'
    print(outfile2)
    outfp = open(outfile2, 'w', newline='')
    w = csv.writer(outfp)

    for name in foo_df.head(50)['gene'].to_list():
        w.writerow(['0', species, name, "(not reviewed)"])
    outfp.close()
            
    

## Export mapping abundance data to a CSV, after merging with `species-genes.csv`

In [ ]:
species_genes_df = (pl.read_csv('../outputs.cds/singleclust/species-genes.csv')
    .filter(pl.col("good") == 1)
    .with_columns(pl.col('gene_name').alias('gene'))
    .select(["anchor", "gene", "species", "description"])
)

species_genes_df

In [ ]:
merge_df = depth_df.join(species_genes_df, on=["species", "gene"], how='inner')
merge_df

In [ ]:
merge_df.write_csv('../outputs.cds/cds3-genes/mapping-coverage.csv')